<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_pharmacy_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *HGU Cocody — Pharmacie Centrale* dans Power BI Desktop. Il est conçu pour toi si :

- tu as déjà manipulé Excel et tu sais ce qu'est une formule,
- tu as installé Power BI Desktop sur ta machine,
- tu comprends ce qu'est une jointure entre deux tables.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |
| 🚀 | L'optimisation ou la variante avancée |

### Le contexte métier

**HGU Cocody** est un Hôpital Général Universitaire d'Abidjan. Sa pharmacie centrale dessert une dizaine de services cliniques (Urgences, Cardiologie, Maternité, Pédiatrie...). Quatre métriques pilotent l'équipe pharmacie :

1. **Coût total de consommation** — combien dépense l'hôpital chaque mois en médicaments
2. **Jours de rupture** — combien de jours par an un médicament critique manque (avec impact patient)
3. **Taux de livraison à temps** — fiabilité des fournisseurs (cible 90 %)
4. **Statut de stock prédictif** — combien de médicaments seront en rupture imminente, à commander cette semaine, à surveiller (modèle ML)

Le dashboard que tu vas construire répond à 4 questions stratégiques :

| Page | Question |
|---|---|
| 1 — Vue executive | Quelle est la santé globale de la pharmacie ce mois-ci ? |
| 2 — Consommation services | Quels services consomment le plus, et de quels médicaments ? |
| 3 — Fournisseurs | Qui livre à temps, qui prend du retard, à quel coût ? |
| 4 — Prévisions & alertes | Qu'est-ce qu'il faut commander dans les 4 prochaines semaines ? |

## 🗺️ Sommaire détaillé

| Partie | Section | Durée |
|---|---|---|
| **I — Fondations** | Sources · Import CSV · Auto Date/Time | 30 min |
| **II — Modélisation** | Étoile · Calendrier · 8 relations · Marquage | 50 min |
| **III — Table `_Mesures`** | Création | 10 min |
| **IV — 27 mesures DAX** | 7 dossiers : KPIs (7) · Variations (2) · Coûts (1) · Prévisions ML (4) · Statut & Alertes (5) · Couleurs (4) · Sous-titres (4) | 2 h 30 |
| **V — Design** | Charte HGU · Mockup PPTX → PNG | 30 min |
| **VI — 4 pages** | Vue executive / Consommation / Fournisseurs / Alertes | 1 h 15 |
| **VII — Finitions** | Slicers · Navigation · Mise en forme conditionnelle | 25 min |
| **VIII — Validation** | Checklist · Pièges · Storytelling · Annexes | 45 min |

**Total** ≈ **6 h 30** de travail effectif.

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — le « grain »

Une ligne de chaque table représente une entité bien identifiée. Avant d'écrire la moindre formule, complète mentalement : *« Une ligne de cette table = un(e) ____ »*.

### Inventaire des 9 sources

| Fichier | Grain | Rôle | Volumétrie |
|---|---|---|---|
| `medicaments.csv` | 1 médicament unique du formulaire | Dimension | ~50 lignes |
| `services.csv` | 1 service hospitalier (Cardio, Urgences...) | Dimension | ~10 lignes |
| `fournisseurs.csv` | 1 fournisseur de médicaments | Dimension | ~5 lignes |
| `consommations.csv` | 1 consommation d'1 médicament par 1 service à 1 date | Fait principal | ~20 000 lignes |
| `commandes_fournisseurs.csv` | 1 commande passée à 1 fournisseur pour 1 médicament | Fait | ~2 500 lignes |
| `ruptures_stock.csv` | 1 épisode de rupture pour 1 médicament (date début → date fin) | Fait | ~150 lignes |
| `previsions_4semaines.csv` | 1 prévision ML pour 1 médicament × 1 semaine (S1 à S4) | Fait dérivé ML | ~200 lignes |
| `stock_securite_optimise.csv` | 1 médicament avec son ROP et EOQ optimisés (sortie ML) | Fait dérivé ML | ~50 lignes |

### ⚠️ Piège fréquent — distinguer historique vs prévisionnel

`consommations`, `commandes_fournisseurs` et `ruptures_stock` sont **historiques** : ils décrivent ce qui s'est passé. `previsions_4semaines` et `stock_securite_optimise` sont **prédictifs** : ils sortent du notebook ML. Ne mélange jamais les deux familles dans une même mesure d'agrégat — utilise les premières pour analyser, les secondes pour décider.

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

L'inconvénient : il faut une connexion internet au premier chargement. Une fois publié sur le service Power BI, le rapport peut être planifié pour rafraîchir tout seul.

### Les 8 URLs à utiliser

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/medicaments.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/services.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/fournisseurs.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/consommations.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/commandes_fournisseurs.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/data/ruptures_stock.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/corrige/previsions_4semaines.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/corrige/stock_securite_optimise.csv
```

> 💡 *Remplace par ton chemin de dépôt réel. Si les CSV sont en local pour l'instant, utilise `Obtenir les données → Texte/CSV` ; le reste de la procédure est identique.*

### ⚠️ Piège type — `cout_euro`

Si la colonne `consommations[cout_euro]` arrive en String avec point décimal (ex: `"152.30"`), force-la en **Nombre décimal** dans Power Query, sinon les `SUM` plantent silencieusement.

> 📸 **Capture ci-dessous** : *éditeur Power Query avec les 9 requêtes nommées dans le volet gauche.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/01_powerquery_9_requetes.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

### ⚠️ Critique pour ce projet

`commandes_fournisseurs` et `ruptures_stock` ont **plusieurs colonnes de date** (date_commande, date_livraison_reelle, date_debut, date_fin). Sans cette désactivation, Power BI crée 4 LocalDateTables fantômes qui pèsent et empêchent les relations propres.

> 📸 **Capture ci-dessous** : *fenêtre Options avec la case « Date/heure automatique » décochée.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# II — Modéliser les données

## 2.1 Le schéma en étoile

### 📘 Concept clé

Schéma en étoile : faits au centre, dimensions autour. Relations 1→N uniquement, OneDirection. C'est le pattern que Power BI optimise.

### Diagramme du modèle HGU Cocody

```
                              +------------------+
                              |   Calendrier     |
                              +--------+---------+
                                       | 1
                                       | N
    +-----------------+        +-------+----------+        +------------------+
    |   medicaments   |1   N   |                  |   N   1|     services     |
    +--------+--------+--------+  consommations   +--------+---------+--------+
             |       |         +------------------+                  |
             |       |                                               |
             | 1     | 1                                             |
             |       |                                               |
           N |     N |                                               |
    +--------+----+ +-+---------------------+    +-----------------+
    | commandes   | | ruptures_stock        |    |  fournisseurs   |
    | fournisseurs|<-| (historique)          |    +--------+--------+
    +-------------+ +----------------------+              | 1
             |                                            |
             | 1 fournisseur (N) ←→ commandes (1)         | N
             +--------------------------------------------+

    +---------------------------+      +--------------------------+
    |   previsions_4semaines    |      | stock_securite_optimise  |
    | (sortie ML, hebdomadaire) |      |  (sortie ML, ROP/EOQ)    |
    +-------------+-------------+      +-------------+------------+
                  | N                                | N
                  +-----------+        +-------------+
                              | 1    1 |
                          +---+--------+---+
                          |   medicaments  |
                          +----------------+
```

## 2.2 Créer la table Calendrier

**Modélisation → Nouvelle table** :

```dax
Calendrier = 
VAR _start = DATE(2023,1,1)
VAR _end   = DATE(2025,12,31)
RETURN
ADDCOLUMNS(
    CALENDAR(_start, _end),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Mois_Nom_Long", FORMAT([Date], "mmmm", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Semaine",       WEEKNUM([Date], 2)
)
```

## 2.3 Établir les 8 relations propres

### Tableau des relations

| # | De (1) | Clé | Vers (N) | Clé | Direction |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `consommations` | `date` | Single |
| 2 | `Calendrier` | `Date` | `commandes_fournisseurs` | `date_commande` | Single |
| 3 | `medicaments` | `id_medicament` | `consommations` | `id_medicament` | Single |
| 4 | `medicaments` | `id_medicament` | `commandes_fournisseurs` | `id_medicament` | Single |
| 5 | `medicaments` | `id_medicament` | `ruptures_stock` | `id_medicament` | Single |
| 6 | `medicaments` | `id_medicament` | `previsions_4semaines` | `id_medicament` | Single |
| 7 | `services` | `id_service` | `consommations` | `id_service` | Single |
| 8 | `fournisseurs` | `id_fournisseur` | `commandes_fournisseurs` | `id_fournisseur` | Single |

**Vue Modèle → glisser la clé** de la table 1 vers la table N. Cardinalité 1:N, direction Simple.

> 📸 **Capture ci-dessous** : *Vue Modèle finale avec les 8 relations visibles autour de la table consommations.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/03_modele_etoile_relations.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

### ⚠️ Piège — `stock_securite_optimise`

Cette table sort du notebook ML et utilise une colonne `medicament` (le **nom**) au lieu de `id_medicament`. Deux options :
1. Idéal : renommer dans le CSV pour utiliser `id_medicament`, puis créer la relation classique
2. Pragmatique : garder la jointure sur `nom`, mais la passer en **Single OneDirection** (M-1, pas 1-1 BothDirections)

Une bidirectionnelle ici crée des ambiguïtés sur les mesures `Nb medicaments en alerte rouge` et `Statut Stock`.

## 2.4 Marquer Calendrier comme table de dates

Vue Données → `Calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**. Sans ça, `PREVIOUSMONTH` retourne blank silencieusement et la mesure `Variation Mensuelle %` est cassée.

> 📸 **Capture ci-dessous** : *boîte de dialogue Marquer comme table de dates avec la colonne Date sélectionnée.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/marquer_table_date.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive.

---
# IV — Construire les 27 mesures DAX

### Vue d'ensemble des 7 dossiers

| Dossier | Mesures | Rôle |
|---|---|---|
| `01 — KPIs Globaux` | 7 | Compteurs et KPIs principaux |
| `02 — Variations` | 2 | Comparaison vs mois précédent |
| `03 — Coûts & Ruptures` | 1 | Estimation économique des ruptures |
| `04 — Prévisions ML` | 4 | Prévisions hebdomadaires + ROP/EOQ |
| `05 — Statut & Alertes` | 5 | Cascade de statut stock + compteurs d'alerte |
| `06 — Couleurs conditionnelles` | 4 | Mise en forme conditionnelle (médicaments, fournisseurs, statut) |
| `07 — Sous-titres dynamiques` | 4 | Texte qui s'adapte aux slicers, 1 par page |

### 📘 Pourquoi des préfixes `01 —`, `02 —` ?

Les dossiers d'affichage Power BI sont triés alphabétiquement. Sans préfixe numéroté, l'ordre serait alphabétique (`Couleurs`, `Coûts`, `KPIs`, `Prévisions`, ...) — illogique. Avec `01 — KPIs Globaux`, `02 — Variations`, etc., l'ordre logique est imposé : on commence par les KPIs de base, on finit par les sous-titres.

## 4.1 Dossier `01 — KPIs Globaux` (7 mesures)

```dax
Total Unites Consommees = SUM(consommations[quantite_consommee])
```
*Format : `#,0`*

```dax
Cout Total Consommation = SUM(consommations[cout_euro])
```
*Format : `#,0.00 " €"`*

```dax
Nb Jours Rupture = 
CALCULATE(
    SUMX(ruptures_stock, ruptures_stock[duree_jours]),
    ruptures_stock[impact_clinique] <> "Impact mineur"
)
```
*On exclut volontairement les ruptures à impact mineur — focus sur ce qui blesse le patient.*

```dax
Patients Affectes Total = SUM(ruptures_stock[patients_affectes])
```

```dax
Taux Livraison a Temps = 
VAR livrees = 
    CALCULATE(
        COUNTROWS(commandes_fournisseurs),
        commandes_fournisseurs[statut] = "Livrée"
    )
VAR a_temps = 
    CALCULATE(
        COUNTROWS(commandes_fournisseurs),
        commandes_fournisseurs[statut] = "Livrée",
        commandes_fournisseurs[retard_jours] = 0
    )
RETURN DIVIDE(a_temps, livrees)
```
*Format : `0.00 %`. Cible métier : 90 %.*

```dax
Retard Moyen Jours = 
AVERAGEX(
    FILTER(commandes_fournisseurs, commandes_fournisseurs[retard_jours] > 0),
    commandes_fournisseurs[retard_jours]
)
```
*On moyenne uniquement les commandes en retard (filter > 0), sinon les commandes à l'heure tirent la moyenne vers 0 et masquent le problème.*

```dax
Nb retard = CALCULATE(COUNTROWS(commandes_fournisseurs), commandes_fournisseurs[retard_jours] = 0)
```

### ⚠️ Piège fréquent — `Retard Moyen Jours`

Sans le `FILTER` qui exclut les retards de 0, `AVERAGE(retard_jours)` te donnerait 0,7 j alors que le vrai problème est 2,5 j sur les commandes en retard. C'est la différence entre « tout va bien en moyenne » et « quand ça déraille, c'est 2,5 j de retard ».

## 4.2 Dossier `02 — Variations` (2 mesures)

```dax
Consommation Mois Precedent = 
CALCULATE(
    [Total Unites Consommees],
    PREVIOUSMONTH(Calendrier[Date])
)
```

```dax
Variation Mensuelle % = 
DIVIDE(
    [Total Unites Consommees] - [Consommation Mois Precedent],
    [Consommation Mois Precedent]
)
```
*Format : `0.00 %`. La consommation se compare au mois précédent (pas à N-1) car le pilotage est mensuel.*

## 4.3 Dossier `03 — Coûts & Ruptures` (1 mesure)

```dax
Cout Ruptures Estime = 
SUMX(
    ruptures_stock,
    ruptures_stock[patients_affectes] * ruptures_stock[duree_jours] * 50
)
```

### 📘 Le facteur 50 €

On multiplie par **50 € par patient × jour** pour estimer le surcoût économique d'une rupture (transfert, retard de soins, achat d'urgence à un autre fournisseur). C'est une approximation, mais elle rend tangible un coût caché.

## 4.4 Dossier `04 — Prévisions ML` (4 mesures)

```dax
Prevision Semaine 1 = 
CALCULATE(
    SUM(previsions_4semaines[quantite_prevue_semaine]),
    previsions_4semaines[semaine] = 1
)
```

```dax
Prevision Semaine 2 = 
CALCULATE(
    SUM(previsions_4semaines[quantite_prevue_semaine]),
    previsions_4semaines[semaine] = 2
)
```

```dax
ROP Recommande = MAX(stock_securite_optimise[point_commande_ROP])
```

```dax
EOQ Recommande = MAX(stock_securite_optimise[qte_commande_EOQ])
```

### 📘 ROP et EOQ — vocabulaire métier

- **ROP** (Reorder Point) : seuil de stock en dessous duquel il faut passer commande maintenant pour éviter la rupture pendant le délai de livraison
- **EOQ** (Economic Order Quantity) : quantité optimale à commander qui minimise le coût total (commande + stockage)

Les deux sortent du notebook ML, qui les calcule par médicament selon l'historique de consommation et le délai fournisseur.

## 4.5 Dossier `05 — Statut & Alertes` (5 mesures)

### La mesure pivot — Statut Stock

```dax
Statut Stock = 
VAR stock_actuel = MAX(medicaments[stock_actuel])
VAR stock_secu   = MAX(medicaments[stock_securite])
VAR rop          = [ROP Recommande]
VAR prev_s1      = [Prevision Semaine 1]
RETURN
    SWITCH(TRUE(),
        stock_actuel < stock_secu,         "RUPTURE IMMINENTE",
        stock_actuel < rop,                "COMMANDER CETTE SEMAINE",
        stock_actuel < rop + prev_s1,      "SURVEILLER",
        "Stock OK"
    )
```

### 📘 La logique en cascade

Quatre niveaux d'alerte, du plus grave au plus calme :
1. **RUPTURE IMMINENTE** : on est sous le stock de sécurité — agir aujourd'hui
2. **COMMANDER CETTE SEMAINE** : on est sous le ROP — passer commande dans les 7 jours
3. **SURVEILLER** : on est entre ROP et ROP + prévision S1 — vigilance
4. **Stock OK** : tout est calme

### Compteurs et valorisation

```dax
Nb medicaments en alerte rouge = 
VAR _result = 
    CALCULATE(
        COUNTROWS(medicaments),
        FILTER(medicaments, [Statut Stock] = "RUPTURE IMMINENTE")
    )
RETURN IF(ISBLANK(_result), 0, _result)
```

```dax
Nb medicaments a commander = 
CALCULATE(
    COUNTROWS(medicaments),
    FILTER(medicaments, [Statut Stock] = "COMMANDER CETTE SEMAINE")
)
```

```dax
Valeur totale a commander = 
SUMX(
    FILTER(medicaments, [Statut Stock] IN {"RUPTURE IMMINENTE","COMMANDER CETTE SEMAINE"}),
    [EOQ Recommande] * MAX(medicaments[prix_unitaire_euro])
)
```

### Fournisseur le plus urgent (mesure complexe)

```dax
Fournisseur le plus urgent = 
VAR table_alertes =
    FILTER(
        medicaments,
        [Statut Stock] IN {"RUPTURE IMMINENTE", "COMMANDER CETTE SEMAINE"}
    )
VAR table_frn =
    ADDCOLUMNS(
        VALUES(fournisseurs[nom]),
        "nb_alertes",
        CALCULATE(
            COUNTROWS(FILTER(medicaments, [Statut Stock] IN {"RUPTURE IMMINENTE", "COMMANDER CETTE SEMAINE"})),
            TREATAS(
                SELECTCOLUMNS(
                    FILTER(commandes_fournisseurs, commandes_fournisseurs[id_fournisseur] = MAX(fournisseurs[id_fournisseur])),
                    "id_medicament", commandes_fournisseurs[id_medicament]
                ),
                medicaments[id_medicament]
            )
        )
    )
RETURN
    MAXX(
        TOPN(1, table_frn, [nb_alertes], DESC),
        fournisseurs[nom]
    )
```

### 📘 Pas-à-pas pour cette mesure

1. On liste les médicaments en alerte (`table_alertes`)
2. Pour chaque fournisseur, on compte combien de **ses** médicaments en alerte (via `TREATAS` qui propage l'identité côté `medicaments`)
3. On prend le top 1 et on retourne son nom

C'est une mesure pivot pour la page Alertes : elle dit *« ce mois-ci, ton point d'action n°1 c'est de relancer SantéPro Cameroun »*.

## 4.6 Dossier `06 — Couleurs conditionnelles` (4 mesures)

### 📘 Concept clé — pourquoi des mesures couleur ?

Au lieu d'utiliser des visuels HTML Content custom, on alimente les visuels **natifs Power BI** avec des mesures qui retournent un code hex. La mise en forme conditionnelle « Mettre en forme par valeur du champ » applique ensuite la couleur dynamiquement. Avantages : visuels plus rapides, plus accessibles, pas de dépendance à un visuel marketplace, et les filtres et drill-down marchent nativement.

### Les 4 mesures couleur

```dax
Couleur Medicament Critique = 
IF(SELECTEDVALUE(medicaments[medicament_critique]) = TRUE(), "#E24B4A", "#185FA5")
```
*Rouge si médicament critique, bleu sinon. Utilisée pour colorer les barres du Top 8 médicaments.*

```dax
Couleur Retard Fournisseur = 
VAR _r = [Retard Moyen Jours]
RETURN SWITCH(TRUE(),
    _r > 3, "#E24B4A",
    _r > 1, "#BA7517",
    "#1D9E75"
)
```
*Rouge > 3j, orange 1-3j, vert ≤ 1j. Utilisée pour colorer les barres « Retard moyen par fournisseur ».*

```dax
Couleur Statut Stock = 
SWITCH([Statut Stock],
    "RUPTURE IMMINENTE",        "#FEE2E2",
    "COMMANDER CETTE SEMAINE",  "#FEF3C7",
    "SURVEILLER",               "#FFFDF0",
    "#FFFFFF"
)
```
*Fond pâle pour le tableau d'alerte, ligne par ligne.*

```dax
Couleur Texte Statut Stock = 
SWITCH([Statut Stock],
    "RUPTURE IMMINENTE",        "#E24B4A",
    "COMMANDER CETTE SEMAINE",  "#BA7517",
    "SURVEILLER",               "#888780",
    "#1D9E75"
)
```
*Couleur texte saturée pour faire ressortir le statut sur le fond pâle.*

### 🔧 Application sur un visuel barre native

1. Insérer un graphique en barres natif
2. Format → **Barres** → cliquer le **fx** à côté de Couleur
3. **Mettre en forme par : Valeur du champ**
4. Choisir la mesure couleur correspondante (`Couleur Medicament Critique` ou `Couleur Retard Fournisseur`)

### 🔧 Application sur un tableau (cellule)

1. Format du tableau → **Cellules** → fx sur **Couleur d'arrière-plan**
2. Mettre en forme par : Valeur du champ → `Couleur Statut Stock`
3. Idem fx sur **Couleur de la police** → `Couleur Texte Statut Stock`

## 4.7 Dossier `07 — Sous-titres dynamiques` (4 mesures)

### 📘 Concept clé — le sous-titre raconte ce qu'on filtre

Sans sous-titre dynamique, l'utilisateur ne sait jamais s'il regarde l'année 2023 entière, le mois de juin, ou un médicament en particulier. Le sous-titre dynamique transforme un visuel muet en visuel narratif. Une mesure par page, branchée sur une **Carte** ou une **Zone de texte** en haut de page.

```dax
Sous Titre Page1 = 
VAR _annee = SELECTEDVALUE(Calendrier[Annee], "Toutes années")
VAR _mois  = SELECTEDVALUE(Calendrier[Mois_Nom], "Tous mois")
VAR _med   = SELECTEDVALUE(medicaments[nom], "Tous medicaments")
RETURN "HGU Cocody  ·  " & _annee & "  ·  " & _mois & "  ·  " & _med
```

```dax
Sous Titre Page2 = 
VAR _annee   = SELECTEDVALUE(Calendrier[Annee], "Toutes années")
VAR _service = SELECTEDVALUE(services[nom_service], "Tous services")
VAR _med     = SELECTEDVALUE(medicaments[nom], "Tous medicaments")
RETURN "Periode : " & _annee & "  ·  Service : " & _service & "  ·  Medicament : " & _med
```

```dax
Sous Titre Page3 = 
VAR _annee = SELECTEDVALUE(Calendrier[Annee], "Toutes années")
VAR _frn   = SELECTEDVALUE(fournisseurs[nom], "Tous fournisseurs")
VAR _nb    = COUNTROWS(FILTER(commandes_fournisseurs, commandes_fournisseurs[statut] = "Livree"))
RETURN "Periode : " & _annee & "  ·  Fournisseur : " & _frn & "  ·  " & _nb & " commandes livrees"
```

```dax
Sous Titre Page4 = 
VAR _med      = SELECTEDVALUE(medicaments[nom], "Tous medicaments")
VAR _nb_rouge = [Nb medicaments en alerte rouge]
VAR _nb_cmd   = CALCULATE(
                    COUNTROWS(FILTER(medicaments, [Statut Stock] IN {"RUPTURE IMMINENTE","COMMANDER CETTE SEMAINE"}))
                )
RETURN "Medicament : " & _med & "  ·  " & _nb_rouge & " ruptures imminentes  ·  " & _nb_cmd & " a commander"
```

### 🔧 Comment l'afficher dans le rapport

Insérer une **Carte** ou une **Zone de texte** sur la page → champ `Sous Titre PageN` → désactiver les bordures → appliquer la police Segoe UI 12 et la couleur grise.

---
# V — Design system

## 5.1 Charte graphique HGU Cocody

Le secteur santé utilise une charte **vert hospitalier** sobre, sans excès visuel.

### Palette

| Rôle | Hex | Usage |
|---|---|---|
| Sidebar foncée | `#0F503C` | Bandeau de navigation |
| Primaire (vert ciel) | `#1D9E75` | Titres, accents positifs, état OK |
| Secondaire (bleu) | `#185FA5` | Médicaments non-critiques, courbes |
| Danger | `#E24B4A` | Médicaments critiques, ruptures, retards > 3j |
| Warning | `#BA7517` | Retards 1-3j, alertes orange |
| Neutre | `#888780` | Labels, axes, gris textuel |
| Fond clair | `#F5F8F6` | Fond de cards |
| Fond page | `#F0F4F8` | Arrière-plan rapport |
| Texte principal | `#2C2C2A` | Hero numbers |

### Typographie

| Élément | Police | Taille | Poids |
|---|---|---|---|
| Titre de page | Segoe UI | 22 | 600 |
| Sous-titre dynamique | Segoe UI | 12 | 400 |
| Hero KPI | Segoe UI | 32 | 700 |
| Label KPI | Segoe UI | 11 | 600 |

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes (cards à ombre, sidebars). La méthode pro :
1. Dessiner la mise en page dans **PowerPoint** (mockup vierge sans données)
2. Exporter en **PNG haute résolution** (1280×720 ou 2001×1125)
3. Importer comme **arrière-plan de page** dans Power BI
4. Poser les visuels Power BI **par-dessus**

### Ce que le mockup PPTX vierge doit contenir

✅ **À inclure :**
- Sidebar de navigation avec onglets (item actif surligné)
- Logo HGU Cocody + DataProjectLab Academy en footer
- Cards / rectangles vides avec border-top accent coloré
- Encadrés vides pour les charts et tableaux

❌ **À NE PAS inclure :**
- Titre de page (zone de texte Power BI dynamique)
- Slicers Année / Mois (segments natifs)
- Bouton « Retour à l'accueil » (bouton natif avec action Navigation)
- Toute valeur de KPI ou texte dans les cards
- Données dans les charts ou tableaux

### 🔧 Méthode 1 — Export PNG depuis PowerPoint (recommandé)

PowerPoint exporte par défaut en 96 DPI. Pour un 150 DPI lisible :

1. **Win + R** → `regedit` → **Entrée**
2. Aller dans `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
3. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)**
4. Nom : `ExportBitmapResolution` — Valeur : `150` (décimal)
5. Fermer regedit, **redémarrer PowerPoint**

Puis : **Fichier** → **Enregistrer sous** → **PNG** → **Toutes les diapositives**.

### 🔧 Méthode 2 — CloudConvert

1. Aller sur [cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png)
2. Charger `mockup_hgu_blank.pptx`
3. Options → 150 DPI → 1280×720
4. **Convert** → télécharger les 4 PNG

### Renommage final

```
bg-01-vue-executive.png
bg-02-consommation-services.png
bg-03-fournisseurs.png
bg-04-previsions-alertes.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page** (icône pinceau au niveau page)
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement de l'image** → **Adapter**
4. **Transparence** → **0 %**

> 📸 **Capture ci-dessous** : *panneau Format de la page avec l'image PNG en arrière-plan.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/05_application_arriere_plan.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VI — Construire les 4 pages

## 6.1 Page 1 — Vue executive

> *« Quelle est la santé globale de la pharmacie ce mois-ci ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Slicers | Liste déroulante | `Calendrier[Annee]`, `Calendrier[Mois_Nom]`, `medicaments[nom]` |
| Sous-titre | Carte | `Sous Titre Page1` |
| KPI 1 | Carte | `Cout Total Consommation` (border-top vert) |
| KPI 2 | Carte | `Nb Jours Rupture` (border-top rouge, sous-titre « Cible : 0-7 ruptures critiques ») |
| KPI 3 | Carte | `Patients Affectes Total` (border-top orange) |
| KPI 4 | Carte | `Taux Livraison a Temps` (border-top rouge, sous-titre « Cible : 90 % · Retard moy 2j ») |
| Évolution mensuelle | Courbes | Axe X : `Calendrier[Annee_Mois]`, Y : `Total Unites Consommees` (vert plein), avec ligne pointillée `Consommation Mois Precedent` |
| Top 8 médicaments | Barres horizontales **natives** | Axe Y : `medicaments[nom]`, Axe X : `Cout Total Consommation`, Top N : 8, **Couleur barre : `Couleur Medicament Critique`** (rouge si critique, bleu sinon) |
| Répartition par catégorie | Donut | `medicaments[categorie]` × `Cout Total Consommation` |

### 🔧 Configurer le Top 8 médicaments en barre native

1. Insérer un **Graphique en barres horizontales**
2. Axe Y : `medicaments[nom]` · Axe X : `Cout Total Consommation`
3. Volet Filtres → Filtres sur ce visuel → `medicaments[nom]` → **Top N** = 8 par `Cout Total Consommation`
4. Format → **Barres** → **fx** sur Couleur → **Mettre en forme par : Valeur du champ** → `Couleur Medicament Critique`
5. Tri descendant sur la valeur

> 📸 **Capture ci-dessous** : *page Vue executive avec barres rouges (médicaments critiques) et bleues (non-critiques) sur le Top 8.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/06_page_vue_executive.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.2 Page 2 — Consommation services

> *« Quels services consomment le plus, et de quels médicaments ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Sous-titre | Carte | `Sous Titre Page2` |
| Évolution mensuelle par service | Barres empilées | Axe X : `Calendrier[Annee_Mois]`, Y empilé : `Total Unites Consommees`, légende : `services[nom_service]` |
| Poids relatif des services | Treemap | `services[nom_service]` × `Total Unites Consommees` |
| Heatmap Services × Médicaments | Matrice | Lignes : `services[nom_service]`, Colonnes : `medicaments[nom]`, Valeur : `Total Unites Consommees`, fond gradient vert |
| Tableau détaillé | Table | `Calendrier[Mois_Nom]`, `medicaments[nom]`, `services[nom_service]`, `Cout Total Consommation`, `Total Unites Consommees` |

### 📘 Comment lire le treemap

Chaque rectangle = un service, taille proportionnelle à sa consommation totale. Tu vois en 2 secondes que la Réanimation pèse plus que la Pédiatrie même si la Réanimation a moins de patients : c'est le coût des médicaments de soins intensifs qui domine.

> 📸 **Capture ci-dessous** : *page Consommation services avec le treemap Réanimation/Oncologie/Chirurgie en haut.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/07_page_consommation_services.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.3 Page 3 — Fournisseurs

> *« Qui livre à temps, qui prend du retard, à quel coût ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Sous-titre | Carte | `Sous Titre Page3` |
| Taux Livraison à Temps | Jauge | `Taux Livraison a Temps`, min 0, max 1, cible 0,90 |
| Fiabilité vs Délai | Nuage de points | X : `Retard Moyen Jours`, Y : `Taux Livraison a Temps` × 100, taille : `Cout Total Consommation`, légende : `fournisseurs[nom]` |
| Timeline commandes livrées | Aires | Axe X : `Calendrier[Annee_Mois]`, Y : `commandes_fournisseurs[montant_eur]` (rempli vert pâle, contour vert) |
| Retard moyen par fournisseur | Barres horizontales **natives** | Axe Y : `fournisseurs[nom]`, Axe X : `Retard Moyen Jours`, **Couleur barre : `Couleur Retard Fournisseur`** (rouge > 3j, orange 1-3j, vert ≤ 1j) |

### 🔧 Configurer le Retard moyen en barre native

1. Insérer un **Graphique en barres horizontales**
2. Axe Y : `fournisseurs[nom]` · Axe X : `Retard Moyen Jours`
3. Format → **Barres** → **fx** sur Couleur → **Valeur du champ** → `Couleur Retard Fournisseur`
4. Tri descendant

### ⚠️ Piège — la jauge

Power BI ne met pas la jauge à l'échelle 0-100 % par défaut. **Format → Jauge → Bornes** : Min = 0, Max = 1 (puisque `Taux Livraison a Temps` est un ratio entre 0 et 1, pas un pourcentage déjà multiplié).

> 📸 **Capture ci-dessous** : *page Fournisseurs avec la jauge à 37 % (en rouge) et le scatter Fiabilité vs Délai.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/08_page_fournisseurs.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.4 Page 4 — Prévisions & alertes

> *« Qu'est-ce qu'il faut commander dans les 4 prochaines semaines ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Sous-titre | Carte | `Sous Titre Page4` |
| KPI 1 | Carte (border-top rouge) | `Nb medicaments en alerte rouge` (« Médicament en alerte rouge ») |
| KPI 2 | Carte (border-top orange) | `Nb medicaments a commander` (« Médicament à commander ») |
| KPI 3 | Carte (border-top vert) | `Fournisseur le plus urgent` (« Fournisseur le plus urgent ») |
| Tableau d'alerte | Table triée par urgence | `medicaments[nom]`, `medicaments[stock_actuel]`, `ROP Recommande`, `Prevision Semaine 1`, `Prevision Semaine 2`, `EOQ Recommande`, `Statut Stock` (fond `Couleur Statut Stock`, texte `Couleur Texte Statut Stock`) |
| Historique + Prévisions | Courbes | Axe X : `Calendrier[Date]`, Y1 : consommation historique, Y2 : `Prevision Semaine 1` (pointillé), bornes basse/haute en bandes |
| Quantités à commander par fournisseur | Barres horizontales | `fournisseurs[nom]` × `Valeur totale a commander` (en €) |

### 📘 Le tableau d'alerte est l'écran le plus important

Cette page est le tableau de bord opérationnel du pharmacien chef. Chaque matin, il l'ouvre, regarde le tableau, et passe les commandes des médicaments en RUPTURE IMMINENTE et COMMANDER CETTE SEMAINE. Le but n'est pas l'analyse — c'est l'action.

### 🔧 Coloration ligne par ligne du tableau

Sur la colonne `Statut Stock` du tableau :
1. Format du visuel → **Cellules** → fx sur **Couleur d'arrière-plan** → **Mettre en forme par : Valeur du champ** → `Couleur Statut Stock`
2. Idem fx sur **Couleur de la police** → `Couleur Texte Statut Stock`

Résultat : RUPTURE IMMINENTE en cellule rouge clair + texte rouge soutenu, COMMANDER CETTE SEMAINE en jaune + orange, SURVEILLER en crème + gris, Stock OK en blanc + vert.

> 📸 **Capture ci-dessous** : *page Prévisions & alertes avec le tableau d'alerte trié par urgence.*

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/pharmacy_analytics/powerbi/tuto/09_page_previsions_alertes.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Slicers, navigation, finitions

**Slicers globaux** sur chaque page : `Calendrier[Annee]`, `Calendrier[Mois_Nom]`, `medicaments[nom]`. Clic droit → **Synchroniser les segments → cocher toutes les pages**.

**Navigation sidebar verte** : utiliser des **boutons natifs** avec action **Navigation de page** pour Vue executive, Consommation services, Fournisseurs, Prévisions & alertes. Le bouton actif a un fond légèrement plus clair (`#1D9E75` au lieu de `#0F503C`).

**Carte « Fournisseur le plus urgent »** sur la page Alertes : ajouter une zone de texte avec border-top vert et le texte « SantéPro Cameroun » en gros (alimenté par la mesure `Fournisseur le plus urgent`), avec « Fournisseur le plus urgent » en sous-titre gris.

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** : 9 tables visibles + Calendrier + _Mesures, 8 relations actives, 0 LocalDateTable, 0 bidirectionnelle inutile · `Calendrier` marquée comme table de dates · Auto Date/Time désactivé.

**Mesures** : 27 dans `_Mesures` · 7 dossiers numérotés `01 — ...` à `07 — ...` · format défini (€, %, nombre) · aucune `/` (toutes en `DIVIDE`).

**Visuels natifs** : Top 8 médicaments en barres horizontales avec `Couleur Medicament Critique` · Retard fournisseurs en barres avec `Couleur Retard Fournisseur` · Tableau d'alerte avec mise en forme conditionnelle `Couleur Statut Stock` + `Couleur Texte Statut Stock`.

**Pages** : 4 pages avec sous-titre dynamique · Slicers Année/Mois/Médicament synchronisés · Navigation sidebar fonctionnelle · Couleurs conformes à la charte verte HGU.

**Performance** : ouverture < 5 s · aucun visuel en erreur.

## 8.2 Pièges fréquents et solutions

| Symptôme | Cause | Correction |
|---|---|---|
| `PREVIOUSMONTH` renvoie blank | `Calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| `Statut Stock` reste à "Stock OK" partout | Relation `stock_securite_optimise` cassée ou bidirectionnelle | Vérifier la relation 1-N OneDirection sur `nom` ou créer une vraie clé `id_medicament` |
| Les barres du Top 8 ne sont pas colorées | Mise en forme conditionnelle absente | Format → Barres → fx sur Couleur → Valeur du champ → `Couleur Medicament Critique` |
| Jauge à 3 700 % au lieu de 37 % | Min/Max mal calibrés | Format Jauge → Bornes : Min 0, Max 1 |
| Le retard moyen est à 0,7j (faux) | `AVERAGE` inclut les commandes à l'heure | Utiliser `AVERAGEX(FILTER(... retard > 0), ...)` |
| Tableau d'alerte n'a pas la couleur de fond | Mise en forme conditionnelle non appliquée | Cellules → fx → Couleur d'arrière-plan → choisir `Couleur Statut Stock` |
| Treemap des services affiche "(blank)" | Relation `services` ↔ `consommations` cassée | Vérifier `id_service` dans Vue Modèle |
| Variation Mensuelle % à 0 systématiquement | Slicer Mois actif filtre aussi le mois précédent | Retirer le slicer Mois sur les visuels qui montrent la variation, ou utiliser `ALL` dans la mesure |

## 8.3 Storytelling exécutif

Pour présenter au directeur médical, suis l'ordre des 4 pages :

1. **Vue executive** : « Sur 2023, on a dépensé 1,63 M€ en médicaments. 260 jours de rupture critique ont impacté 2 633 patients. Le taux de livraison à temps n'est qu'à 37 %. »
2. **Consommation services** : « La Réanimation et l'Oncologie pèsent à elles deux ~35 % de la consommation. C'est là qu'il faut être le plus précis sur le stock. »
3. **Fournisseurs** : « Tous nos fournisseurs sont en retard moyen entre 1,8 et 2,2 jours. Aucun ne respecte le SLA de 0 jour de retard. C'est notre problème n°1. »
4. **Prévisions & alertes** : « Cette semaine : 0 médicament en rupture imminente, mais 18 à commander. Fournisseur prioritaire : SantéPro Cameroun. »
5. **Recommandation** : renégocier le SLA avec les 5 fournisseurs (passer de retard moyen 2j à < 1j) → projection : -50 % de jours de rupture sur 12 mois.

## 8.4 Annexes

### Règles DAX universelles à graver

1. Toujours `DIVIDE` (jamais `/`)
2. Pattern `VAR ... RETURN` même pour les mesures simples
3. `AVERAGE` ignore les blanks mais pas les zéros — utiliser `AVERAGEX(FILTER(...))` pour exclure les zéros
4. Pour `IN {"valeur1","valeur2"}` dans un FILTER, syntax DAX moderne (équivalent à OR multiples)
5. `TREATAS` propage l'identité d'une colonne d'une table vers une autre — utile pour les jointures virtuelles
6. Mesures couleur (`Couleur ...`) qui retournent un code hex → branchées sur les visuels natifs via fx → Valeur du champ. Plus performant et plus accessible que des visuels HTML Content custom.

### Mapping mockup PPTX ↔ pages Power BI

| Slide PPTX | Background PNG | Page Power BI |
|---|---|---|
| 1 | `bg-01-vue-executive.png` | Vue executive |
| 2 | `bg-02-consommation-services.png` | Consommation services |
| 3 | `bg-03-fournisseurs.png` | Fournisseurs |
| 4 | `bg-04-previsions-alertes.png` | Prévisions & alertes |

### Livrables finaux du projet

- `HGU - Rapport Power BI.pbix`
- `mockup_hgu.pptx` (avec données factices)
- `mockup_hgu_blank.pptx` (vierge pour exporter en PNG)
- 4 fichiers `bg-XX-*.png`
- `Synthese_Projet_Sante_HGU.pptx` (synthèse exécutive)
- `recommandations_hgu.docx` (recommandations chiffrées)

---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">HGU Cocody — Pharmacy Analytics</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>